# Pandas Data Cleaning & Reshaping — Catch-Up Notebook

This covers the full class workflow in order:

1. Build messy data, find missing values & duplicates
2. Drop duplicates
3. Clean text columns (`.str.strip()`, `.str.title()`)
4. Safely coerce bad numbers/dates to `NaN` / `NaT`
5. Impute missing study hours with the **median**
6. `.groupby()` + aggregate functions
7. `.transform()` to compare a row against its group average
8. `pd.concat()` to stack DataFrames vertically
9. `pivot()` / `melt()` to reshape wide ↔ long

**Run every cell in order.** Exercises are at the bottom.

In [3]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 120)
print("pandas", pd.__version__)

pandas 3.0.3


---
## 1. Create messy sample data

Real data is dirty. We deliberately plant every problem we're about to fix:

| Problem | Where you see it |
|---|---|
| Extra whitespace | `"  ali raza "` |
| Inconsistent casing | `"SARA KHAN"` vs `"bilal ahmed"` |
| Numbers stored as text | `"5.5"` (a string, not a float) |
| Garbage values | `"abc"` in a numeric column |
| Mixed date formats | `2024-01-15` vs `2024/02/20` |
| Real missing values | `None` |
| Duplicate rows | Ali Raza and Sara Khan appear twice |

In [4]:
rawdata = {
    "name":        [" ali raza ", "SARA KHAN", "bilal ahmed", "SARA KHAN", "zainab  ", "Hamza Iqbal", "  ali raza "],
    "city":        ["lahore ", "Karachi", "  lahore", "Karachi", "ISLAMABAD", "karachi ", "lahore "],
    "study_hours": ["5.5", "8", "abc", "8", None, "3.5", "5.5"],
    "score":       [72, 91, 64, 91, 88, 55, 72],
    "join_date":   ["2024-01-15", "2024/02/20", "not a date", "2024/02/20", "2024-03-01", None, "2024-01-15"],
}

df = pd.DataFrame(rawdata)
df

,name,city,study_hours,score,join_date
0,ali raza,lahore,5.5,72,2024-01-15
1,SARA KHAN,Karachi,8,91,2024/02/20
2,bilal ahmed,lahore,abc,64,not a date
3,SARA KHAN,Karachi,8,91,2024/02/20
4,zainab,ISLAMABAD,NaN,88,2024-03-01
5,Hamza Iqbal,karachi,3.5,55,NaN
6,ali raza,lahore,5.5,72,2024-01-15


### Inspect before you touch anything

Three commands you should run on *every* new dataset:

- `df.info()` — row count, column dtypes, non-null counts
- `df.isna().sum()` — missing values per column
- `df.duplicated().sum()` — how many rows are exact copies of an earlier row

Look closely at the `info()` output: `study_hours` is `object` (text), not a number.
That means you **cannot** do maths on it yet.

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   name         7 non-null      str  
 1   city         7 non-null      str  
 2   study_hours  6 non-null      str  
 3   score        7 non-null      int64
 4   join_date    6 non-null      str  
dtypes: int64(1), str(4)
memory usage: 610.0 bytes


In [6]:
print("Missing values per column:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Missing values per column:
name           0
city           0
study_hours    1
score          0
join_date      1
dtype: int64

Duplicate rows: 1


`df.duplicated()` returns a boolean Series — `True` means "this row already appeared above".
Use `keep=False` to flag **all** copies (including the first), which is what you want when you're *looking* at them.

In [7]:
df[df.duplicated(keep=False)]

,name,city,study_hours,score,join_date
1,SARA KHAN,Karachi,8,91,2024/02/20
3,SARA KHAN,Karachi,8,91,2024/02/20


---
## 2. Drop duplicates

`drop_duplicates()` keeps the first occurrence by default.

After dropping rows, the index has gaps (0, 1, 2, 4, 5). `.reset_index(drop=True)` renumbers it 0..n-1.
`drop=True` throws the old index away instead of turning it into a column — forget it and you get a stray `index` column.

In [8]:
print("Before:", df.shape)

df = df.drop_duplicates().reset_index(drop=True)

print("After: ", df.shape)
df

Before: (7, 5)
After:  (6, 5)


,name,city,study_hours,score,join_date
0,ali raza,lahore,5.5,72,2024-01-15
1,SARA KHAN,Karachi,8,91,2024/02/20
2,bilal ahmed,lahore,abc,64,not a date
3,zainab,ISLAMABAD,NaN,88,2024-03-01
4,Hamza Iqbal,karachi,3.5,55,NaN
5,ali raza,lahore,5.5,72,2024-01-15


**Useful variant:** `df.drop_duplicates(subset=["name"], keep="last")` — treat rows as duplicates
based only on certain columns, and keep the last one instead of the first.

---
## 3. Clean the text columns

The `.str` accessor applies a string method to every value in a Series, and the calls **chain**:

```python
df["name"].str.strip().str.title()
```

- `.str.strip()` → removes leading/trailing whitespace
- `.str.title()` → `"sara khan"` → `"Sara Khan"`
- other handy ones: `.str.lower()`, `.str.upper()`, `.str.replace()`, `.str.contains()`

**Why this matters:** `"lahore "` and `"Lahore"` are different strings. Without this step your
`groupby("city")` later would produce 4 separate "Lahore" groups instead of 1.

In [9]:
print("Unique cities BEFORE cleaning:", df["city"].unique())

df["name"] = df["name"].str.strip().str.title()
df["city"] = df["city"].str.strip().str.title()

print("Unique cities AFTER cleaning: ", df["city"].unique())
df

Unique cities BEFORE cleaning: <ArrowStringArray>
['lahore ', 'Karachi', '  lahore', 'ISLAMABAD', 'karachi ']
Length: 5, dtype: str
Unique cities AFTER cleaning:  <ArrowStringArray>
['Lahore', 'Karachi', 'Islamabad']
Length: 3, dtype: str


,name,city,study_hours,score,join_date
0,Ali Raza,Lahore,5.5,72,2024-01-15
1,Sara Khan,Karachi,8,91,2024/02/20
2,Bilal Ahmed,Lahore,abc,64,not a date
3,Zainab,Islamabad,NaN,88,2024-03-01
4,Hamza Iqbal,Karachi,3.5,55,NaN
5,Ali Raza,Lahore,5.5,72,2024-01-15


---
## 4. Safely coerce bad numbers and dates

This is the key idea of the lesson. `study_hours` contains `"abc"`. If you try to force the whole
column to a number, pandas raises an error and you get nothing.

`errors="coerce"` says: **convert what you can, and turn anything unconvertible into a missing value.**
The bad data becomes a `NaN` you can deal with later, instead of a crash.

- `pd.to_numeric(series, errors="coerce")` → bad values become `NaN`
- `pd.to_datetime(series, errors="coerce")` → bad values become `NaT` (Not a Time)

`format="mixed"` lets `to_datetime` handle `2024-01-15` and `2024/02/20` in the same column.

In [10]:
df["study_hours"] = pd.to_numeric(df["study_hours"], errors="coerce")
df["join_date"]   = pd.to_datetime(df["join_date"], errors="coerce", format="mixed")

print(df.dtypes)
df

name                      str
city                      str
study_hours           float64
score                   int64
join_date      datetime64[us]
dtype: object


,name,city,study_hours,score,join_date
0,Ali Raza,Lahore,5.5,72,2024-01-15
1,Sara Khan,Karachi,8.0,91,2024-02-20
2,Bilal Ahmed,Lahore,NaN,64,NaT
3,Zainab,Islamabad,NaN,88,2024-03-01
4,Hamza Iqbal,Karachi,3.5,55,NaT
5,Ali Raza,Lahore,5.5,72,2024-01-15


Now recheck missing values. Notice the count went **up** — that's expected and correct.
`"abc"` and `"not a date"` were always junk; coercion just made that visible and countable.

In [11]:
df.isna().sum()

name           0
city           0
study_hours    2
score          0
join_date      2
dtype: int64

---
## 5. Impute missing study hours with the median

`fillna(value)` replaces missing values.

**Why median, not mean?** The median is the middle value, so a single extreme outlier
(someone logging 200 hours) barely moves it, while it would drag the mean badly.
For skewed real-world data, median is the safer default.

Note the assign-back pattern — `df["col"] = df["col"].fillna(...)`. In modern pandas,
`df["col"].fillna(x, inplace=True)` silently does nothing on a slice, so always assign back.

In [12]:
median_hours = df["study_hours"].median()
print("Median study hours:", median_hours)
print("Mean for comparison:", round(df["study_hours"].mean(), 2))

df["study_hours"] = df["study_hours"].fillna(median_hours)

print("\nMissing values now:")
print(df.isna().sum())
df

Median study hours: 5.5
Mean for comparison: 5.62

Missing values now:
name           0
city           0
study_hours    0
score          0
join_date      2
dtype: int64


,name,city,study_hours,score,join_date
0,Ali Raza,Lahore,5.5,72,2024-01-15
1,Sara Khan,Karachi,8.0,91,2024-02-20
2,Bilal Ahmed,Lahore,5.5,64,NaT
3,Zainab,Islamabad,5.5,88,2024-03-01
4,Hamza Iqbal,Karachi,3.5,55,NaT
5,Ali Raza,Lahore,5.5,72,2024-01-15


`join_date` still has `NaT`. That's a deliberate choice: you can't meaningfully invent a date the
way you can a typical value. Options would be to leave it, drop those rows, or forward-fill —
it depends on the question you're answering.

---
## 6. Group and aggregate with `.groupby()`

`groupby("city")` splits the table into one mini-table per city, applies a function to each, and
combines the results. This is the **split–apply–combine** pattern.

The clearest syntax is **named aggregation**:

```python
df.groupby("city").agg(
    new_column_name=("column_to_use", "function")
)
```

Common functions: `"mean"`, `"sum"`, `"count"`, `"min"`, `"max"`, `"median"`, `"std"`, `"nunique"`.

In [13]:
city_stats = df.groupby("city").agg(
    students    = ("name", "count"),
    avg_score   = ("score", "mean"),
    max_score   = ("score", "max"),
    total_hours = ("study_hours", "sum"),
).reset_index()

city_stats

,city,students,avg_score,max_score,total_hours
0,Islamabad,1,88.000000,88,5.5
1,Karachi,2,73.000000,91,11.5
2,Lahore,3,69.333333,72,16.5


Two other syntaxes you'll meet:

In [14]:
# Single column, single function -> returns a Series
print(df.groupby("city")["score"].mean())

print()

# Dictionary style -> multiple functions, gives messy multi-level column names
print(df.groupby("city").agg({"score": ["mean", "max"], "study_hours": "sum"}))

city
Islamabad    88.000000
Karachi      73.000000
Lahore       69.333333
Name: score, dtype: float64

               score     study_hours
                mean max         sum
city                                
Islamabad  88.000000  88         5.5
Karachi    73.000000  91        11.5
Lahore     69.333333  72        16.5


---
## 7. `.transform()` — compare each row to its group

`.agg()` **shrinks** the data: 5 rows → 3 cities.

`.transform()` **keeps the original shape**: it computes the group value and then broadcasts it back
to every row in that group. That's what lets you put a group statistic next to an individual row
and compare them.

| | Input rows | Output rows |
|---|---|---|
| `.agg("mean")` | 5 | 3 (one per city) |
| `.transform("mean")` | 5 | 5 (city average repeated) |

In [15]:
df["city_avg_score"] = df.groupby("city")["score"].transform("mean")

df[["name", "city", "score", "city_avg_score"]]

,name,city,score,city_avg_score
0,Ali Raza,Lahore,72,69.333333
1,Sara Khan,Karachi,91,73.000000
2,Bilal Ahmed,Lahore,64,69.333333
3,Zainab,Islamabad,88,88.000000
4,Hamza Iqbal,Karachi,55,73.000000
5,Ali Raza,Lahore,72,69.333333


Now that both values sit on the same row, a plain comparison gives a boolean flag column:

In [16]:
df["above_city_avg"] = df["score"] > df["city_avg_score"]

df[["name", "city", "score", "city_avg_score", "above_city_avg"]]

,name,city,score,city_avg_score,above_city_avg
0,Ali Raza,Lahore,72,69.333333,True
1,Sara Khan,Karachi,91,73.000000,True
2,Bilal Ahmed,Lahore,64,69.333333,False
3,Zainab,Islamabad,88,88.000000,False
4,Hamza Iqbal,Karachi,55,73.000000,False
5,Ali Raza,Lahore,72,69.333333,True


Boolean columns are handy: `True` counts as 1, so `.sum()` counts them and `.mean()` gives a proportion.

In [17]:
print("Students above their city average:", df["above_city_avg"].sum())
print("Proportion:", round(df["above_city_avg"].mean(), 2))

# Filtering with the flag
df[df["above_city_avg"]]

Students above their city average: 3
Proportion: 0.5


,name,city,study_hours,score,join_date,city_avg_score,above_city_avg
0,Ali Raza,Lahore,5.5,72,2024-01-15,69.333333,True
1,Sara Khan,Karachi,8.0,91,2024-02-20,73.000000,True
5,Ali Raza,Lahore,5.5,72,2024-01-15,69.333333,True


---
## 8. Stack DataFrames vertically with `pd.concat()`

`pd.concat([df1, df2])` stacks rows on top of each other. Columns are matched **by name**;
any column missing from one side becomes `NaN`.

The gotcha: concat **preserves the original indexes**, so you end up with repeated index labels
(0,1,2,3,4,0,1). Fix it with `ignore_index=True` (or `.reset_index(drop=True)` afterwards).

In [18]:
batch2 = pd.DataFrame({
    "name":        ["Omar Sheikh", "Ayesha Malik"],
    "city":        ["Lahore", "Islamabad"],
    "study_hours": [6.0, 7.5],
    "score":       [79, 84],
    "join_date":   pd.to_datetime(["2024-04-10", "2024-04-12"]),
})

base = df[["name", "city", "study_hours", "score", "join_date"]]

# WRONG - duplicated index labels
bad = pd.concat([base, batch2])
print("Index without ignore_index:", bad.index.tolist())

# RIGHT
combined = pd.concat([base, batch2], ignore_index=True)
print("Index with ignore_index:   ", combined.index.tolist())

combined

Index without ignore_index: [0, 1, 2, 3, 4, 5, 0, 1]
Index with ignore_index:    [0, 1, 2, 3, 4, 5, 6, 7]


,name,city,study_hours,score,join_date
0,Ali Raza,Lahore,5.5,72,2024-01-15
1,Sara Khan,Karachi,8.0,91,2024-02-20
2,Bilal Ahmed,Lahore,5.5,64,NaT
3,Zainab,Islamabad,5.5,88,2024-03-01
4,Hamza Iqbal,Karachi,3.5,55,NaT
5,Ali Raza,Lahore,5.5,72,2024-01-15
6,Omar Sheikh,Lahore,6.0,79,2024-04-10
7,Ayesha Malik,Islamabad,7.5,84,2024-04-12


`axis=1` concatenates side by side (adding columns) instead of stacking rows — but for joining on a
key column you almost always want `pd.merge()` rather than `concat`.

---
## 9. Reshape: `pivot()` (long → wide) and `melt()` (wide → long)

**Long format** — one row per observation. Good for storage, plotting, and groupby.

**Wide format** — one row per subject, one column per variable. Good for reading and comparing.

`pivot()` needs three things: `index` (rows), `columns` (what becomes new column headers),
`values` (what fills the cells).

In [19]:
long_df = pd.DataFrame({
    "student": ["Ali", "Ali", "Ali", "Sara", "Sara", "Sara"],
    "subject": ["Math", "Physics", "English", "Math", "Physics", "English"],
    "marks":   [88, 74, 91, 95, 80, 67],
})

long_df

,student,subject,marks
0,Ali,Math,88
1,Ali,Physics,74
2,Ali,English,91
3,Sara,Math,95
4,Sara,Physics,80
5,Sara,English,67


In [20]:
wide = long_df.pivot(index="student", columns="subject", values="marks").reset_index()
wide.columns.name = None   # cosmetic: removes the leftover "subject" label above the index

wide

,student,English,Math,Physics
0,Ali,91,88,74
1,Sara,67,95,80


`melt()` is the exact inverse. It collapses many columns into two: one holding the old column names,
one holding the values.

- `id_vars` — columns to **keep as they are** (the identifier)
- `var_name` — name for the column of old headers
- `value_name` — name for the column of values

In [21]:
back_to_long = wide.melt(id_vars="student", var_name="subject", value_name="marks")

back_to_long.sort_values(["student", "subject"]).reset_index(drop=True)

,student,subject,marks
0,Ali,English,91
1,Ali,Math,88
2,Ali,Physics,74
3,Sara,English,67
4,Sara,Math,95
5,Sara,Physics,80


### `pivot()` vs `pivot_table()`

`pivot()` **fails** if an index/column pair appears more than once — it doesn't know which value to use.
`pivot_table()` handles that by aggregating (default `aggfunc="mean"`).

Rule of thumb: use `pivot()` for pure reshaping, `pivot_table()` when duplicates exist or you want a summary.

In [22]:
long_df.pivot_table(index="student", columns="subject", values="marks", aggfunc="mean")

subject,English,Math,Physics
student,,,
Ali,91.0,88.0,74.0
Sara,67.0,95.0,80.0


---
## Quick reference

```python
# Inspect
df.info(); df.describe(); df.isna().sum(); df.duplicated().sum()

# Clean
df = df.drop_duplicates().reset_index(drop=True)
df["col"] = df["col"].str.strip().str.title()
df["num"] = pd.to_numeric(df["num"], errors="coerce")
df["dt"]  = pd.to_datetime(df["dt"], errors="coerce", format="mixed")
df["num"] = df["num"].fillna(df["num"].median())

# Group
df.groupby("key").agg(new=("col", "mean")).reset_index()   # shrinks
df.groupby("key")["col"].transform("mean")                 # keeps shape

# Combine & reshape
pd.concat([a, b], ignore_index=True)
df.pivot(index="i", columns="c", values="v")
df.melt(id_vars="i", var_name="c", value_name="v")
```

**The three things most likely to be asked in a viva or quiz:**

1. What does `errors="coerce"` do? → Converts unparseable values to `NaN`/`NaT` instead of raising an error.
2. `.agg()` vs `.transform()`? → `agg` returns one row per group; `transform` returns one row per original row.
3. Why `reset_index(drop=True)`? → Renumbers the index after dropping/concatenating rows; `drop=True` discards the old index instead of keeping it as a column.

---
# Exercises

Try these yourself first. Solutions are in the last cell — don't peek.

In [23]:
practice = pd.DataFrame({
    "employee":   [" john doe ", "MARIA GARCIA", "john doe", "Ahmed Ali", "MARIA GARCIA", "li wei "],
    "department": ["sales ", "IT", "SALES", " it", "IT", "Hr"],
    "salary":     ["50000", "72000", "50000", "invalid", "72000", "45000"],
    "years":      [3, 7, 3, 5, 7, None],
    "hire_date":  ["2021-03-01", "2017/06/15", "2021-03-01", "2019-01-20", "2017/06/15", "bad date"],
})

practice

,employee,department,salary,years,hire_date
0,john doe,sales,50000,3.0,2021-03-01
1,MARIA GARCIA,IT,72000,7.0,2017/06/15
2,john doe,SALES,50000,3.0,2021-03-01
3,Ahmed Ali,it,invalid,5.0,2019-01-20
4,MARIA GARCIA,IT,72000,7.0,2017/06/15
5,li wei,Hr,45000,NaN,bad date


**Q1.** How many duplicate rows are there? Drop them and reset the index.

**Q2.** Clean `employee` and `department` (strip whitespace, title case). Confirm `department` has exactly 3 unique values.

> **Hint / bonus:** after Q2 you'll notice two John Doe rows that are now identical but *weren't*
> exact duplicates before cleaning (they differed by whitespace and casing). This is a real trap —
> in messy data it's often worth running `drop_duplicates()` a second time *after* standardising text.

**Q3.** Convert `salary` to numeric and `hire_date` to datetime, coercing bad values.

**Q4.** Fill missing `salary` and `years` with the median of each column.

**Q5.** Build a department summary: headcount, average salary, max years.

**Q6.** Add a `dept_avg_salary` column using `.transform()`, then a boolean `above_dept_avg`.

**Q7.** Concat this new hire and reset the index:
```python
new_hire = pd.DataFrame({"employee": ["Sana Tariq"], "department": ["It"],
                         "salary": [68000.0], "years": [4.0],
                         "hire_date": pd.to_datetime(["2023-08-01"])})
```

**Q8.** Reshape `sales_long` below into wide (quarters as columns), then melt it back.
```python
sales_long = pd.DataFrame({
    "region":  ["North","North","South","South"],
    "quarter": ["Q1","Q2","Q1","Q2"],
    "revenue": [100, 120, 90, 95],
})
```

In [ ]:
# Your work here
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   name            6 non-null      str           
 1   city            6 non-null      str           
 2   study_hours     6 non-null      float64       
 3   score           6 non-null      int64         
 4   join_date       4 non-null      datetime64[us]
 5   city_avg_score  6 non-null      float64       
 6   above_city_avg  6 non-null      bool          
dtypes: bool(1), datetime64[us](1), float64(2), int64(1), str(2)
memory usage: 522.0 bytes


---
### Solutions

In [25]:
# Q1
print("Duplicates:", practice.duplicated().sum())
p = practice.drop_duplicates().reset_index(drop=True)

# Q2
p["employee"]   = p["employee"].str.strip().str.title()
p["department"] = p["department"].str.strip().str.title()
print("Departments:", p["department"].unique())

# Q3
p["salary"]    = pd.to_numeric(p["salary"], errors="coerce")
p["hire_date"] = pd.to_datetime(p["hire_date"], errors="coerce", format="mixed")

# Q4
p["salary"] = p["salary"].fillna(p["salary"].median())
p["years"]  = p["years"].fillna(p["years"].median())
print(p)

# Q5
summary = p.groupby("department").agg(
    headcount  = ("employee", "count"),
    avg_salary = ("salary", "mean"),
    max_years  = ("years", "max"),
).reset_index()
print("\n", summary)

# Q6
p["dept_avg_salary"] = p.groupby("department")["salary"].transform("mean")
p["above_dept_avg"]  = p["salary"] > p["dept_avg_salary"]
print("\n", p[["employee", "department", "salary", "dept_avg_salary", "above_dept_avg"]])

# Q7
new_hire = pd.DataFrame({"employee": ["Sana Tariq"], "department": ["It"],
                         "salary": [68000.0], "years": [4.0],
                         "hire_date": pd.to_datetime(["2023-08-01"])})
all_staff = pd.concat([p.drop(columns=["dept_avg_salary", "above_dept_avg"]), new_hire],
                      ignore_index=True)
print("\n", all_staff)

# Q8
sales_long = pd.DataFrame({
    "region":  ["North","North","South","South"],
    "quarter": ["Q1","Q2","Q1","Q2"],
    "revenue": [100, 120, 90, 95],
})
sales_wide = sales_long.pivot(index="region", columns="quarter", values="revenue").reset_index()
sales_wide.columns.name = None
print("\n", sales_wide)
print("\n", sales_wide.melt(id_vars="region", var_name="quarter", value_name="revenue"))

Duplicates: 1
Departments: <ArrowStringArray>
['Sales', 'It', 'Hr']
Length: 3, dtype: str
       employee department   salary  years  hire_date
0      John Doe      Sales  50000.0    3.0 2021-03-01
1  Maria Garcia         It  72000.0    7.0 2017-06-15
2      John Doe      Sales  50000.0    3.0 2021-03-01
3     Ahmed Ali         It  50000.0    5.0 2019-01-20
4        Li Wei         Hr  45000.0    4.0        NaT

   department  headcount  avg_salary  max_years
0         Hr          1     45000.0        4.0
1         It          2     61000.0        7.0
2      Sales          2     50000.0        3.0

        employee department   salary  dept_avg_salary  above_dept_avg
0      John Doe      Sales  50000.0          50000.0           False
1  Maria Garcia         It  72000.0          61000.0            True
2      John Doe      Sales  50000.0          50000.0           False
3     Ahmed Ali         It  50000.0          61000.0           False
4        Li Wei         Hr  45000.0          4500